In [2]:
# ============================================
# Setup: imports + SparkSession (local/YARN auto-detect)
# Author: Sulaiman Alhammad (ID: 230103)
# ============================================
import os
import sys

# Detect environment once
is_cluster = os.environ.get("SPARK_MODE") == "yarn"

if not is_cluster:
    # Pin the worker Python to the kernel Python (Windows + PySpark needs this).
    os.environ.setdefault("PYSPARK_PYTHON", sys.executable)
    os.environ.setdefault("PYSPARK_DRIVER_PYTHON", sys.executable)
    # Java 17+ removed the security-manager API Spark uses internally;
    # re-enable the legacy behaviour. No-op on the cluster.
    if "PYSPARK_SUBMIT_ARGS" not in os.environ:
        os.environ["PYSPARK_SUBMIT_ARGS"] = (
            "--conf spark.driver.extraJavaOptions=-Djava.security.manager=allow "
            "--conf spark.executor.extraJavaOptions=-Djava.security.manager=allow "
            "pyspark-shell"
        )

import time
import random
from datetime import datetime, timedelta

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, hour, to_timestamp, desc, count, sum as Fsum, when
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, BooleanType

builder = SparkSession.builder.appName("M2_Malqa")
builder = builder.master("yarn") if is_cluster else builder.master("local[*]")

spark = builder.getOrCreate()
spark.sparkContext.setLogLevel("WARN")

print(f"Spark version: {spark.version}")
print(f"Master:        {spark.sparkContext.master}")

Spark version: 4.0.2
Master:        local[*]


In [3]:
# ============================================
# Data loader: HDFS -> local sample -> synthetic generator
# Author: Sulaiman Alhammad (ID: 230103)
# ============================================
def generate_synthetic(spark, n=10_000, seed=42):
    """Realistic Chicago-crime rows for local testing. Distributions
    chosen to roughly track the M1 results so Phase A top-10s match."""
    rng = random.Random(seed)
    crime_types = [("THEFT", 0.205), ("BATTERY", 0.192), ("CRIMINAL DAMAGE", 0.115),
                   ("NARCOTICS", 0.094), ("ASSAULT", 0.068), ("BURGLARY", 0.060),
                   ("MOTOR VEHICLE THEFT", 0.050), ("DECEPTIVE PRACTICE", 0.046),
                   ("ROBBERY", 0.040), ("OTHER OFFENSE", 0.130)]
    locations   = [("STREET", 0.31), ("RESIDENCE", 0.17), ("APARTMENT", 0.08),
                   ("SIDEWALK", 0.06), ("OTHER", 0.04), ("PARKING LOT", 0.05),
                   ("RETAIL STORE", 0.04), ("RESTAURANT", 0.03),
                   ("BAR OR TAVERN", 0.02), ("COMMERCIAL", 0.20)]
    arrest_rate = {"NARCOTICS": 0.90, "ASSAULT": 0.40, "BATTERY": 0.35,
                   "ROBBERY": 0.30, "OTHER OFFENSE": 0.30,
                   "DECEPTIVE PRACTICE": 0.15, "BURGLARY": 0.12,
                   "CRIMINAL DAMAGE": 0.10, "THEFT": 0.10,
                   "MOTOR VEHICLE THEFT": 0.08}

    def pick(items):
        r, cum = rng.random(), 0.0
        for v, p in items:
            cum += p
            if r < cum:
                return v
        return items[-1][0]

    rows = []
    base = datetime(2024, 1, 1)
    for i in range(n):
        ct = pick(crime_types)
        loc = pick(locations)
        ts = base + timedelta(days=rng.randint(0, 364),
                              hours=rng.randint(0, 23),
                              minutes=rng.randint(0, 59))
        rows.append((
            10_000_000 + i,
            ts.strftime("%m/%d/%Y %I:%M:%S %p"),
            ct, loc,
            rng.random() < arrest_rate.get(ct, 0.20),
            rng.random() < 0.13,
            rng.randint(1, 25),
            ts.year,
        ))
    schema = StructType([
        StructField("ID", IntegerType(), False),
        StructField("Date", StringType(), False),
        StructField("Primary Type", StringType(), False),
        StructField("Location Description", StringType(), False),
        StructField("Arrest", BooleanType(), False),
        StructField("Domestic", BooleanType(), False),
        StructField("District", IntegerType(), False),
        StructField("Year", IntegerType(), False),
    ])
    return spark.createDataFrame(rows, schema)


def load_or_generate(spark):
    for path in ("hdfs:///data/chicago_crimes.csv", "data/chicago_crimes_sample.csv"):
        try:
            df = spark.read.option("header", True).option("inferSchema", True).csv(path)
            n = df.count()
            if n >= 1000:
                print(f"Loaded {n:,} rows from {path}")
                return df
            print(f"{path} has only {n} rows -- too small, falling through")
        except Exception as e:
            print(f"Could not load {path}: {type(e).__name__}")
    print("Generating 10,000 synthetic rows for local testing")
    return generate_synthetic(spark, n=10_000)


df_raw = load_or_generate(spark)
df_raw.printSchema()
print(f"Total rows: {df_raw.count():,}")

Could not load hdfs:///data/chicago_crimes.csv: Py4JJavaError
Could not load data/chicago_crimes_sample.csv: AnalysisException
Generating 10,000 synthetic rows for local testing
root
 |-- ID: integer (nullable = false)
 |-- Date: string (nullable = false)
 |-- Primary Type: string (nullable = false)
 |-- Location Description: string (nullable = false)
 |-- Arrest: boolean (nullable = false)
 |-- Domestic: boolean (nullable = false)
 |-- District: integer (nullable = false)
 |-- Year: integer (nullable = false)

Total rows: 10,000


In [3]:
# ============================================
# Preprocessing: parse Hour, cast Arrest -> label, cast Domestic -> string, drop nulls
# Author: Sulaiman Alhammad (ID: 230103)
# ============================================
df = (df_raw
      .withColumn("Hour", hour(to_timestamp(col("Date"), "MM/dd/yyyy hh:mm:ss a")))
      .withColumn("Hour", when(col("Hour").isNull(), 12).otherwise(col("Hour")))
      .withColumn("label", col("Arrest").cast("integer"))
      .withColumn("Domestic", col("Domestic").cast("string"))   # StringIndexer needs string/numeric, not boolean
      .na.drop(subset=["Primary Type", "Domestic", "District", "Hour", "label"]))

df.select("Date", "Hour", "Arrest", "label", "Primary Type", "Domestic", "District").show(5, truncate=False)
print(f"Rows after preprocessing: {df.count():,}")

In [3]:
# ============================================
# Task 1: Crime Type Distribution (DataFrame API)
# Author: Sulaiman Alhammad (ID: 230103)
# ============================================
top_types = df.groupBy("Primary Type").count().orderBy(desc("count"))
top_types.show(10, truncate=False)

print("\nM1 MapReduce reference (full 793,074-row dataset, top 5):")
print("  THEFT           162,688")
print("  BATTERY         151,930")
print("  CRIMINAL DAMAGE  91,241")
print("  NARCOTICS        74,127")
print("  ASSAULT          54,070")

In [ ]:
# ============================================
# Task 2: Location Hotspots (Spark SQL)
# Author: Fayez Algosaibi (ID: 230092)
# ============================================
df.createOrReplaceTempView("crimes")
top_locations = spark.sql("""
    SELECT `Location Description` AS location, COUNT(*) AS total
    FROM crimes
    GROUP BY `Location Description`
    ORDER BY total DESC
    LIMIT 10
""")
top_locations.show(10, truncate=False)

print("
M1 MapReduce reference (top 5):")
print("  STREET    248,326")
print("  RESIDENCE 136,393")
print("  APARTMENT  61,235")
print("  SIDEWALK   47,506")
print("  OTHER      29,671")

In [ ]:
# ============================================
# Task 3: Crime Trend Over Years
# Author: Abdullah Bin Salamah (ID: 220690)
# ============================================
yearly = df.groupBy("Year").count().orderBy("Year")
yearly.show(30)

try:
    import matplotlib.pyplot as plt
    pdf = yearly.toPandas()
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(pdf["Year"], pdf["count"], marker="o")
    ax.set_xlabel("Year")
    ax.set_ylabel("Crime count")
    ax.set_title("Chicago crimes per year")
    plt.tight_layout()
    plt.show()
except Exception as e:
    print(f"(skipping chart: {e})")

print("\nM1 MapReduce reference (top 5 years by count):")
print("  2001  467,301")
print("  2002  205,267")
print("  2023   81,461")
print("  2025   12,710")
print("  2022    4,678")

In [ ]:
# ============================================
# Task 4: Arrest rate overall + per crime type
# Author: Abdullah Bin Salamah (ID: 220690)
# ============================================
overall = df.agg(Fsum(col("label")).alias("arrests"),
                 count("*").alias("total")).collect()[0]
arr, tot = overall["arrests"], overall["total"]
print(f"Overall arrest rate: {arr:,} / {tot:,} = {arr/tot*100:.2f}%")

per_type = (df.groupBy("Primary Type")
              .agg(Fsum(col("label")).alias("arrests"), count("*").alias("total"))
              .withColumn("arrest_rate_pct", (col("arrests") / col("total") * 100))
              .orderBy(desc("total")))
per_type.show(10, truncate=False)

print("\nM1 MapReduce reference (full dataset):")
print("  False (Not Arrested):  571,140  (72.0%)")
print("  True  (Arrested):      221,932  (28.0%)")
print("\nInterpretation: NARCOTICS has the highest arrest rate -- officers")
print("typically witness the offense directly. THEFT and MOTOR VEHICLE THEFT")
print("have the lowest rates because crimes are reported after the suspect")
print("has already left the scene.")

In [ ]:
# ============================================
# Task 5: Feature Engineering Pipeline
# Author: Saleh Alkhattaf (ID: 230381)
# ============================================
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, VectorAssembler

crime_idx = StringIndexer(inputCol="Primary Type", outputCol="crime_index",   handleInvalid="skip")
dom_idx   = StringIndexer(inputCol="Domestic",     outputCol="domestic_index", handleInvalid="skip")
assembler = VectorAssembler(
    inputCols=["District", "crime_index", "Hour", "domestic_index"],
    outputCol="features",
)

feature_pipeline = Pipeline(stages=[crime_idx, dom_idx, assembler])
prepared = (feature_pipeline.fit(df).transform(df)
            .select("features", "label", "Primary Type", "Domestic", "District", "Hour"))

print("Feature vector layout: [District, crime_index, Hour, domestic_index]")
prepared.show(5, truncate=False)

train, test = prepared.randomSplit([0.8, 0.2], seed=42)
train.cache(); test.cache()
print(f"Train rows: {train.count():,}    Test rows: {test.count():,}")

In [ ]:
# ============================================
# Task 6: Train + evaluate LR / RF / GBT
# Author: Saleh Alkhattaf (ID: 230381)
# ============================================
from pyspark.ml.classification import LogisticRegression, RandomForestClassifier, GBTClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator
import pandas as pd


def evaluate(name, model_cls, **params):
    t0 = time.time()
    model = model_cls(labelCol="label", featuresCol="features", **params).fit(train)
    train_s = time.time() - t0
    pred = model.transform(test)
    auc = BinaryClassificationEvaluator(labelCol="label", metricName="areaUnderROC").evaluate(pred)
    acc = MulticlassClassificationEvaluator(labelCol="label", metricName="accuracy").evaluate(pred)
    f1  = MulticlassClassificationEvaluator(labelCol="label", metricName="f1").evaluate(pred)
    pre = MulticlassClassificationEvaluator(labelCol="label", metricName="weightedPrecision").evaluate(pred)
    rec = MulticlassClassificationEvaluator(labelCol="label", metricName="weightedRecall").evaluate(pred)
    cm = pred.groupBy("label", "prediction").count().collect()
    m  = {(int(r["label"]), int(r["prediction"])): r["count"] for r in cm}
    return {
        "model": name,
        "AUC": round(auc, 4), "Accuracy": round(acc, 4), "F1": round(f1, 4),
        "Precision": round(pre, 4), "Recall": round(rec, 4),
        "TN": m.get((0, 0), 0), "FP": m.get((0, 1), 0),
        "FN": m.get((1, 0), 0), "TP": m.get((1, 1), 0),
        "Train_sec": round(train_s, 2),
    }, model


lr_res,  lr_model  = evaluate("LogisticRegression", LogisticRegression,     maxIter=100, regParam=0.01)
rf_res,  rf_model  = evaluate("RandomForest",       RandomForestClassifier, numTrees=100, maxDepth=5)
gbt_res, gbt_model = evaluate("GBT",                GBTClassifier,          maxIter=50,  maxDepth=5)

results = [lr_res, rf_res, gbt_res]
comparison = pd.DataFrame(results)
print("\n=== Model comparison ===")
print(comparison.to_string(index=False))